In [1]:
import os, sys, time, json, shutil
from pathlib import Path
PROJECT = Path(r"C:\Users\hp\Documents\teliport_prototype")
os.chdir(PROJECT / "notebooks")
sys.path.insert(0, str(PROJECT / "src"))
%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd
from cmapss_data import load_all, SUBSETS

DATA_DIR = PROJECT / "data" / "raw" / "CMAPSS"
MODELS = PROJECT / "models"
REPORTS = PROJECT / "reports"
WO_DIR = PROJECT / "work_orders"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("Working dir:", Path.cwd())

Working dir: C:\Users\hp\Documents\teliport_prototype\notebooks


In [2]:
%%writefile ../src/inference.py
"""Fleet scoring: raw sensor history -> one snapshot per engine (RUL + health + root cause)."""
from __future__ import annotations

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from features import last_cycle
from modeling import predict


def load_models(fd, models_dir):
    models_dir = Path(models_dir)
    return (joblib.load(models_dir / f"preprocessor_{fd}.joblib"),
            joblib.load(models_dir / f"lgbm_{fd}.joblib"),
            joblib.load(models_dir / f"health_monitor_{fd}.joblib"))


def score_fleet(raw_df, fd, models_dir, model_version="lgbm_v1"):
    """raw_df = sensor history (unit, cycle, settings, sensors) for any number of engines."""
    prep, rul_model, hm = load_models(fd, models_dir)
    feats = prep.transform(raw_df)
    flags = hm.transform(feats)
    last = last_cycle(feats)
    last_flags = flags.groupby("unit").tail(1).set_index("unit")
    snap = pd.DataFrame({
        "subset": fd,
        "unit": last["unit"].to_numpy(),
        "cycles_seen": last["cycle"].astype(int).to_numpy(),
        "pred_RUL": np.round(predict(rul_model, last, prep.feature_names_), 1),
    })
    snap["anomaly_score"] = np.round(last_flags.loc[snap["unit"], "anomaly_score"].to_numpy(), 2)
    snap["alarm"] = last_flags.loc[snap["unit"], "alarm"].to_numpy().astype(bool)
    snap["top_sensors"] = [hm.top_contributors(row) for _, row in last.iterrows()]
    snap["model_version"] = model_version
    return snap

Writing ../src/inference.py


In [3]:
%%writefile ../src/triage_agent.py
"""Rule-based Triage Agent (no LLM).

Reads model outputs for each engine, decides severity with transparent threshold rules,
flags exceptions for a human, and writes maintenance work orders (JSON + CSV log).
"""
from __future__ import annotations

import json
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

SEVERITY_ORDER = {"LOW": 0, "MEDIUM": 1, "HIGH": 2, "CRITICAL": 3}

SENSOR_COMPONENT = {
    "s2": "Low-pressure compressor (LPC)", "s3": "High-pressure compressor (HPC)",
    "s4": "Low-pressure turbine (LPT)", "s7": "High-pressure compressor (HPC)",
    "s8": "Fan", "s9": "Engine core (HPC/HPT spool)", "s11": "High-pressure compressor (HPC)",
    "s12": "High-pressure compressor (HPC)", "s13": "Fan", "s14": "Engine core (HPC/HPT spool)",
    "s15": "Fan / bypass duct", "s17": "Bleed air system",
    "s20": "High-pressure turbine (HPT) cooling", "s21": "Low-pressure turbine (LPT) cooling",
}


@dataclass
class TriageConfig:
    critical_rul: float = 20      # <= 20 cycles left  -> CRITICAL
    high_rul: float = 40          # <= 40              -> HIGH
    medium_rul: float = 75        # <= 75              -> MEDIUM, else LOW (monitor only)
    due_fraction: float = 0.5     # schedule work at half the predicted life (2x safety factor)
    disagreement_rul: float = 100 # alarm while RUL > 100 -> models disagree -> human review
    min_cycles: int = 20          # shorter history -> low confidence -> human review
    min_severity_for_wo: str = "MEDIUM"


@dataclass
class Decision:
    subset: str
    unit: int
    cycles_seen: int
    pred_RUL: float
    anomaly_score: float
    alarm: bool
    severity: str
    suspected_component: str
    top_sensors: list
    needs_human_review: bool
    review_reasons: list = field(default_factory=list)
    rules_fired: list = field(default_factory=list)
    model_version: str = "unknown"


class TriageAgent:
    def __init__(self, config: TriageConfig | None = None, out_dir=None):
        self.cfg = config or TriageConfig()
        self.out_dir = Path(out_dir) if out_dir else None

    # ---------- 1. decide ----------
    def assess(self, snap: dict) -> Decision:
        c, rul, alarm = self.cfg, float(snap["pred_RUL"]), bool(snap["alarm"])
        rules, reasons = [], []

        if rul <= c.critical_rul:
            sev = "CRITICAL"; rules.append(f"R1 pred_RUL {rul:.0f} <= {c.critical_rul:.0f}")
        elif rul <= c.high_rul:
            sev = "HIGH"; rules.append(f"R2 pred_RUL {rul:.0f} <= {c.high_rul:.0f}")
        elif rul <= c.medium_rul:
            sev = "MEDIUM"; rules.append(f"R3 pred_RUL {rul:.0f} <= {c.medium_rul:.0f}")
        else:
            sev = "LOW"; rules.append(f"R4 pred_RUL {rul:.0f} > {c.medium_rul:.0f} (healthy)")

        if alarm and sev == "LOW":
            sev = "MEDIUM"; rules.append("E1 anomaly alarm active -> escalate LOW to MEDIUM (inspect)")

        if alarm and rul > c.disagreement_rul:
            reasons.append("Health monitor alarm but RUL model sees > "
                           f"{c.disagreement_rul:.0f} cycles: possible sensor fault or unseen failure mode")
        if rul <= c.high_rul and not alarm:
            reasons.append("RUL model predicts near-failure but no anomaly alarm: verify sensors/data")
        if int(snap["cycles_seen"]) < c.min_cycles:
            reasons.append(f"Only {int(snap['cycles_seen'])} cycles of history: low-confidence prediction")
        rules += [f"X{i+1} human review: {r}" for i, r in enumerate(reasons)]

        top = list(snap.get("top_sensors") or [])
        weights = {}
        for s, _, pct in top:
            comp = SENSOR_COMPONENT.get(s, "Unknown")
            weights[comp] = weights.get(comp, 0) + pct
        component = max(weights, key=weights.get) if weights else "Unknown"

        return Decision(subset=str(snap["subset"]), unit=int(snap["unit"]),
                        cycles_seen=int(snap["cycles_seen"]), pred_RUL=round(rul, 1),
                        anomaly_score=round(float(snap["anomaly_score"]), 2), alarm=alarm,
                        severity=sev, suspected_component=component,
                        top_sensors=[{"sensor": s, "name": n, "share_%": p} for s, n, p in top],
                        needs_human_review=bool(reasons), review_reasons=reasons, rules_fired=rules,
                        model_version=str(snap.get("model_version", "unknown")))

    # ---------- 2. act ----------
    def recommended_action(self, d: Decision, due: int) -> str:
        comp = d.suspected_component
        if d.severity == "CRITICAL":
            return f"Remove from service at next stop; inspect and repair/replace {comp} within {due} cycles."
        if d.severity == "HIGH":
            return f"Schedule maintenance on {comp} within {due} cycles; reserve parts and crew now."
        return f"Plan borescope inspection of {comp} within {due} cycles; increase monitoring frequency."

    def create_work_order(self, d: Decision, now: datetime | None = None) -> dict | None:
        if SEVERITY_ORDER[d.severity] < SEVERITY_ORDER[self.cfg.min_severity_for_wo]:
            return None
        now = now or datetime.now(timezone.utc)
        due = max(1, int(d.pred_RUL * self.cfg.due_fraction))
        wo = {
            "work_order_id": f"WO-{d.subset}-U{d.unit:03d}-C{d.cycles_seen:03d}",
            "created_at": now.isoformat(timespec="seconds"),
            "asset": f"{d.subset}/engine-{d.unit:03d}",
            "severity": d.severity,
            "status": "PENDING_HUMAN_REVIEW" if d.needs_human_review else "AUTO_APPROVED",
            "due_within_cycles": due,
            "recommended_action": self.recommended_action(d, due),
            **{k: v for k, v in asdict(d).items() if k not in ("subset", "unit")},
        }
        if self.out_dir:
            self.out_dir.mkdir(parents=True, exist_ok=True)
            (self.out_dir / f"{wo['work_order_id']}.json").write_text(json.dumps(wo, indent=2))
        return wo

    # ---------- 3. run over a fleet snapshot ----------
    def run(self, snapshots: pd.DataFrame) -> pd.DataFrame:
        rows, orders = [], []
        for snap in snapshots.to_dict("records"):
            d = self.assess(snap)
            wo = self.create_work_order(d)
            rows.append({"subset": d.subset, "unit": d.unit, "cycles_seen": d.cycles_seen,
                         "pred_RUL": d.pred_RUL, "anomaly_score": d.anomaly_score, "alarm": d.alarm,
                         "severity": d.severity, "suspected_component": d.suspected_component,
                         "needs_human_review": d.needs_human_review,
                         "work_order_id": wo["work_order_id"] if wo else None,
                         "wo_status": wo["status"] if wo else "NO_ACTION",
                         "due_within_cycles": wo["due_within_cycles"] if wo else None})
            if wo:
                orders.append(wo)
        decisions = pd.DataFrame(rows)
        if self.out_dir and orders:
            log = pd.DataFrame([{k: (json.dumps(v) if isinstance(v, (list, dict)) else v)
                                 for k, v in o.items()} for o in orders])
            log_path = self.out_dir / "work_order_log.csv"
            if log_path.exists():                       # idempotent: never log the same order twice
                seen = set(pd.read_csv(log_path, usecols=["work_order_id"])["work_order_id"])
                log = log[~log["work_order_id"].isin(seen)]
            if len(log):
                log.to_csv(log_path, mode="a", header=not log_path.exists(), index=False)
        return decisions

Writing ../src/triage_agent.py


In [4]:
from triage_agent import TriageAgent, TriageConfig, SEVERITY_ORDER

agent = TriageAgent()
base = dict(subset="FD001", unit=1, cycles_seen=150, anomaly_score=0.5, alarm=False,
            top_sensors=[("s11", "HPC static pressure (Ps30)", 60.0), ("s4", "LPT outlet temperature (T50)", 40.0)])
cases = [
    ("near failure + alarm",             dict(pred_RUL=8,   alarm=True,  anomaly_score=9.0), "CRITICAL", False),
    ("high risk + alarm",                dict(pred_RUL=35,  alarm=True,  anomaly_score=3.0), "HIGH",     False),
    ("medium risk + alarm",              dict(pred_RUL=60,  alarm=True,  anomaly_score=1.5), "MEDIUM",   False),
    ("healthy, no alarm",                dict(pred_RUL=118),                                  "LOW",      False),
    ("healthy RUL but alarm (90)",       dict(pred_RUL=90,  alarm=True,  anomaly_score=1.8), "MEDIUM",   False),
    ("models disagree (alarm, RUL 120)", dict(pred_RUL=120, alarm=True,  anomaly_score=4.0), "MEDIUM",   True),
    ("near failure but NO alarm",        dict(pred_RUL=15,  alarm=False),                     "CRITICAL", True),
    ("short history",                    dict(pred_RUL=50,  alarm=True,  cycles_seen=12),     "MEDIUM",   True),
    ("exact boundary RUL=20",            dict(pred_RUL=20,  alarm=True),                      "CRITICAL", False),
    ("exact boundary RUL=75",            dict(pred_RUL=75),                                   "MEDIUM",   False),
]
rows = []
for name, over, exp_sev, exp_review in cases:
    d = agent.assess({**base, **over})
    ok = (d.severity == exp_sev) and (d.needs_human_review == exp_review)
    rows.append({"case": name, "severity": d.severity, "expected": exp_sev,
                 "review": d.needs_human_review, "expected_review": exp_review,
                 "component": d.suspected_component, "PASS": ok})
tests = pd.DataFrame(rows)
assert tests["PASS"].all(), "Some agent rule tests failed!"
print(f"All {len(tests)} agent rule tests passed.")
tests

All 10 agent rule tests passed.


,case,severity,expected,review,expected_review,component,PASS
0,near failure + alarm,CRITICAL,CRITICAL,False,False,High-pressure compressor (HPC),True
1,high risk + alarm,HIGH,HIGH,False,False,High-pressure compressor (HPC),True
2,medium risk + alarm,MEDIUM,MEDIUM,False,False,High-pressure compressor (HPC),True
3,"healthy, no alarm",LOW,LOW,False,False,High-pressure compressor (HPC),True
4,healthy RUL but alarm (90),MEDIUM,MEDIUM,False,False,High-pressure compressor (HPC),True
5,"models disagree (alarm, RUL 120)",MEDIUM,MEDIUM,True,True,High-pressure compressor (HPC),True
6,near failure but NO alarm,CRITICAL,CRITICAL,True,True,High-pressure compressor (HPC),True
7,short history,MEDIUM,MEDIUM,True,True,High-pressure compressor (HPC),True
8,exact boundary RUL=20,CRITICAL,CRITICAL,False,False,High-pressure compressor (HPC),True
9,exact boundary RUL=75,MEDIUM,MEDIUM,False,False,High-pressure compressor (HPC),True


In [5]:
from inference import score_fleet

shutil.rmtree(WO_DIR, ignore_errors=True)            # fresh demo run
t0 = time.time()
data = load_all(DATA_DIR)
snaps = pd.concat([score_fleet(data[fd][1], fd, MODELS) for fd in SUBSETS], ignore_index=True)

agent = TriageAgent(out_dir=WO_DIR)
decisions = agent.run(snaps)
decisions.to_csv(REPORTS / "triage_decisions_test.csv", index=False)

print(f"Scored + triaged {len(decisions)} engines in {time.time() - t0:.1f}s")
print("Work-order files written:", len(list(WO_DIR.glob("WO-*.json"))))
print(decisions["wo_status"].value_counts(), "\n")
pd.crosstab(decisions["subset"], decisions["severity"]).reindex(columns=["CRITICAL", "HIGH", "MEDIUM", "LOW"])

Scored + triaged 707 engines in 4.5s
Work-order files written: 348
wo_status
NO_ACTION               359
AUTO_APPROVED           344
PENDING_HUMAN_REVIEW      4
Name: count, dtype: int64 



severity,CRITICAL,HIGH,MEDIUM,LOW
subset,,,,
FD001,11,17,22,50
FD002,50,27,58,124
FD003,13,9,33,45
FD004,39,28,41,140


In [6]:
crit = decisions[(decisions["severity"] == "CRITICAL") & ~decisions["needs_human_review"]].iloc[0]
print(json.dumps(json.loads((WO_DIR / f"{crit.work_order_id}.json").read_text()), indent=2))

rev = decisions[decisions["needs_human_review"] & decisions["work_order_id"].notna()]
print(f"\n{len(rev)} work orders sent to a human. Example reason:")
print(json.loads((WO_DIR / f"{rev.iloc[0].work_order_id}.json").read_text())["review_reasons"])

{
  "work_order_id": "WO-FD001-U020-C184",
  "created_at": "2026-10-30T12:37:00+00:00",
  "asset": "FD001/engine-020",
  "severity": "CRITICAL",
  "status": "AUTO_APPROVED",
  "due_within_cycles": 6,
  "recommended_action": "Remove from service at next stop; inspect and repair/replace High-pressure compressor (HPC) within 6 cycles.",
  "cycles_seen": 184,
  "pred_RUL": 13.7,
  "anomaly_score": 7.92,
  "alarm": true,
  "suspected_component": "High-pressure compressor (HPC)",
  "top_sensors": [
    {
      "sensor": "s11",
      "name": "HPC static pressure (Ps30)",
      "share_%": 20.5
    },
    {
      "sensor": "s7",
      "name": "HPC outlet pressure (P30)",
      "share_%": 12.9
    },
    {
      "sensor": "s13",
      "name": "Corrected fan speed (NRf)",
      "share_%": 12.6
    }
  ],
  "needs_human_review": false,
  "review_reasons": [],
  "rules_fired": [
    "R1 pred_RUL 14 <= 20"
  ],
  "model_version": "lgbm_v1"
}

4 work orders sent to a human. Example reason:
['Health m

In [7]:
truth = pd.concat([data[fd][2].assign(subset=fd) for fd in SUBSETS])[["subset", "unit", "RUL"]]
ev = decisions.merge(truth, on=["subset", "unit"])
cfg = TriageConfig()
band = lambda r: ("CRITICAL" if r <= cfg.critical_rul else "HIGH" if r <= cfg.high_rul
                  else "MEDIUM" if r <= cfg.medium_rul else "LOW")
ev["true_severity"] = ev["RUL"].apply(band)
SO = SEVERITY_ORDER
order = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]

summary = {
    "engines": len(ev),
    "exact_severity_match_%": round(100 * (ev.severity == ev.true_severity).mean(), 1),
    "within_one_level_%": round(100 * ((ev.severity.map(SO) - ev.true_severity.map(SO)).abs() <= 1).mean(), 1),
    "truly_critical_engines": int((ev.RUL <= cfg.critical_rul).sum()),
    "critical_missed (got MEDIUM/LOW)": int(((ev.RUL <= cfg.critical_rul) & (ev.severity.map(SO) <= 1)).sum()),
    "under_triaged_%": round(100 * (ev.severity.map(SO) < ev.true_severity.map(SO)).mean(), 1),
    "over_triaged_%": round(100 * (ev.severity.map(SO) > ev.true_severity.map(SO)).mean(), 1),
    "work_orders_failing_before_due": int((ev.work_order_id.notna() & (ev.RUL < ev.due_within_cycles)).sum()),
    "human_review_rate_%": round(100 * ev.needs_human_review.mean(), 1),
}
pd.Series(summary).to_csv(REPORTS / "agent_evaluation.csv")
for k, v in summary.items():
    print(f"{k:36s} {v}")

print("\nConfusion matrix (rows = true severity, cols = agent severity):")
pd.crosstab(ev["true_severity"], ev["severity"]).reindex(index=order, columns=order, fill_value=0)

engines                              707
exact_severity_match_%               87.6
within_one_level_%                   99.9
truly_critical_engines               123
critical_missed (got MEDIUM/LOW)     0
under_triaged_%                      5.1
over_triaged_%                       7.4
work_orders_failing_before_due       1
human_review_rate_%                  0.8

Confusion matrix (rows = true severity, cols = agent severity):


severity,CRITICAL,HIGH,MEDIUM,LOW
true_severity,,,,
CRITICAL,113,10,0,0
HIGH,0,65,7,0
MEDIUM,0,5,101,19
LOW,0,1,46,340
